# Task 2 - SWAP test

In [5]:
import numpy as np
import pandas as pd
from IPython.display import display

DTYPE = np.complex128
I = np.eye(2, dtype=DTYPE)
H = np.array([[1, 1], [1, -1]], dtype=DTYPE) / np.sqrt(2)

def normalize(psi):
    psi = np.asarray(psi, dtype=DTYPE)
    norm = np.linalg.norm(psi)
    if norm == 0:
        raise ValueError('Cannot normalize the zero vector.')
    return psi / norm

P0 = np.array([[1, 0], [0, 0]], dtype=DTYPE)
P1 = np.array([[0, 0], [0, 1]], dtype=DTYPE)
SWAP = np.array([[1, 0, 0, 0],
                 [0, 0, 1, 0],
                 [0, 1, 0, 0],
                 [0, 0, 0, 1]], dtype=DTYPE)
CSWAP = np.kron(P0, np.eye(4)) + np.kron(P1, SWAP)
H_output = np.kron(H, np.kron(I, I))

## Circuit

The ordering is $|output\ A\ B\rangle$, with output as the most significant qubit. We prepare $|0\rangle|\psi\rangle|\phi\rangle$ and apply H to output, then CSWAP, then H again.

CSWAP leaves the inputs unchanged when output is 0 and swaps them when output is 1. After the circuit, indices 4–7 correspond to output 1. We sum their probabilities and use $F=1-2P(1)$.

In [6]:
def swap_test(psi, phi):
    psi = normalize(psi)
    phi = normalize(phi)
    if psi.shape != (2,) or phi.shape != (2,):
        raise ValueError('Each input must be a one-qubit state.')

    state = np.kron(np.array([1, 0]), np.kron(psi, phi))
    state = H_output @ state
    state = CSWAP @ state
    state = H_output @ state
    assert np.isclose(np.linalg.norm(state), 1)

    prob_1 = np.sum(np.abs(state[4:])**2)
    F_swap = 1 - 2*prob_1
    F_exact = abs(np.vdot(psi, phi))**2
    return prob_1, F_swap, F_exact

## Test cases

### Methodology

We choose three input pairs: 0 vs 0 for identical states, 0 vs 1 for orthogonal states, and + vs 0 for partial overlap. Their expected fidelities are 1, 0 and 0.5 respectively. The circuit stays the same for every pair; only the input states change.

For each pair, we calculate the fidelity in three ways:

1. From the exact final-state probabilities, using $F=1-2P(1)$.
2. From the direct inner product, $F=|\langle\psi|\phi\rangle|^2$, to check the circuit result.
3. From 10,000 simulated measurements of the output qubit, replacing $P(1)$ with the observed fraction of ones.

We sample only the output bit because it contains the information needed for this estimate. The NumPy generator is initialized once with seed 7 and used for all three pairs. The table separates the exact calculations from the sampled estimate, so their differences can be inspected directly.

In [7]:
ket0 = np.array([1, 0], dtype=DTYPE)
ket1 = np.array([0, 1], dtype=DTYPE)
plus = np.array([1, 1], dtype=DTYPE) / np.sqrt(2)

cases = [('0 vs 0', ket0, ket0),
         ('0 vs 1', ket0, ket1),
         ('+ vs 0', plus, ket0)]

shots = 10_000
rng = np.random.default_rng(7)
results = []
for name, psi, phi in cases:
    p1, F_swap, F_exact = swap_test(psi, phi)
    # Sample only the output qubit.
    outcomes = rng.choice([0, 1], size=shots, p=[1-p1, p1])
    F_sampled = 1 - 2*np.mean(outcomes)
    results.append([name, p1, F_swap, F_exact, F_sampled])
    assert np.isclose(F_swap, F_exact)

display(pd.DataFrame(results, columns=['Pair', 'P(1)', 'F SWAP',
                                     'F inner product', 'F sampled']).round(4))
assert np.allclose([row[1] for row in results], [0, 0.5, 0.25])

,Pair,P(1),F SWAP,F inner product,F sampled
0,0 vs 0,0.00,1.0,1.0,1.0000
1,0 vs 1,0.50,0.0,0.0,-0.0134
2,+ vs 0,0.25,0.5,0.5,0.4778


## Results and discussion

The exact circuit fidelities match the direct inner products for all three pairs. The output-1 probabilities are 0, 0.5 and 0.25, giving fidelities of 1, 0 and 0.5.

For identical inputs, swapping the states makes no change. After the last H, the output-1 branch cancels, so every measurement gives 0 and the sampled fidelity is exactly 1.

For orthogonal inputs, both output values have probability 0.5. A finite sample does not have to contain exactly as many zeros as ones. Here the estimated fidelity is -0.0134. Its theoretical standard error is $2\sqrt{0.5(1-0.5)/10000}=0.01$, so this result is 1.34 standard errors below zero. The negative estimate is a sampling fluctuation, not a negative physical fidelity.

For partial overlap, the estimate is 0.4778 instead of 0.5. The standard error is $2\sqrt{0.25(1-0.25)/10000}\approx0.00866$, so the deviation is about 2.56 standard errors. This is a larger fluctuation in this run. The exact circuit calculation still gives 0.5; increasing the number of shots reduces the typical sampling uncertainty.

The relation between overlap and measurement follows from the output-1 branch, which is proportional to $|\psi\rangle|\phi\rangle-|\phi\rangle|\psi\rangle$. Its squared norm gives $P(1)=(1-|\langle\psi|\phi\rangle|^2)/2$. A single zero cannot establish equality, because unequal inputs can also produce that outcome.

The lecture SWAP-test circuit includes an extra NOT at the end. That reverses the output labels, giving $F=2P(1)-1$. The circuit implemented here ends with H and uses $F=1-2P(1)$.